# VisionQuery on Google Colab

Natural-language video search and object recoloring (YOLO-World · CLIP · SAM 2).

1. **Runtime → Change runtime type → T4 GPU**, then save.
2. Run the cells in order (Shift+Enter). The first run takes ~3–5 minutes.
3. Cell 5 prints a link: open it, upload a video, search, recolor.

**بالعربي:** من القائمة اختار Runtime ثم Change runtime type ثم T4 GPU. بعدها شغّل الخلايا بالترتيب، وآخر خلية تطلع لك رابط تفتحه وتستخدم الواجهة.

## 1 · Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > T4 GPU"

## 2 · Get the code

In [ ]:
import os
REPO = "https://github.com/aass11111/visionquery.git"
if os.path.isdir("/content/visionquery/.git"):
    !git -C /content/visionquery pull -q
else:
    !git clone -q {REPO} /content/visionquery
%cd /content/visionquery
!git log --oneline -1

## 3 · Install Python packages
PyTorch, OpenCV and ffmpeg are already on Colab; this only adds what is missing.

In [ ]:
!pip install -q "ultralytics>=8.3.50,<8.4" "git+https://github.com/ultralytics/CLIP.git" lap python-multipart "fastapi>=0.115" "uvicorn[standard]"
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 4 · Build the web interface

In [ ]:
%cd /content/visionquery/frontend
!node --version || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y -qq nodejs)
!npm ci --no-audit --no-fund --loglevel=error && npm run build --silent
!ls dist
%cd /content/visionquery

## 5 · Start the server and open the app

In [ ]:
import os, subprocess, time, urllib.request

env = dict(os.environ,
           FRONTEND_DIST="/content/visionquery/frontend/dist",
           MODEL_DIR="/content/models", DATA_DIR="/content/data")
if "server" in globals() and server.poll() is None:
    server.terminate(); server.wait()
log = open("/content/server.log", "w")
server = subprocess.Popen(["python", "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
                          cwd="/content/visionquery/backend", env=env, stdout=log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print(urllib.request.urlopen("http://localhost:8000/api/health").read().decode()); break
    except Exception:
        time.sleep(1)
else:
    print(open("/content/server.log").read()[-3000:])

from google.colab.output import eval_js
print("\nOpen VisionQuery:", eval_js("google.colab.kernel.proxyPort(8000)"))

### Optional · public link with Cloudflare
Use this if the link above does not open, or to share it from your phone. Anyone with the link can use it while the notebook runs.

In [ ]:
!test -f /content/cloudflared || wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
import subprocess, re, time
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                          stderr=subprocess.PIPE, text=True)
for line in tunnel.stderr:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("Public link:", m.group(0)); break

## Tools

In [ ]:
# Server log (run when something fails)
!tail -n 40 /content/server.log

In [ ]:
# Unit tests (recoloring, mask maths, colour naming for suggestions)
!cd /content/visionquery/backend && python -m pytest -q tests

In [ ]:
# Automatic smoke test: makes a short clip, then runs search -> preview -> recolor through the API
!bash /content/visionquery/scripts/e2e.sh